### Step 1: Retrieve PDB IDs from RCSB Search

This step retrieves PDB IDs from the RCSB database based on user-defined search criteria and downloads the corresponding FASTA files.

#### Functionality
- Parse RCSB search URL containing query conditions  
- Extract PDB IDs that match the search criteria  
- Send requests to RCSB API to retrieve results  
- Download corresponding FASTA sequence files  

#### Inputs
- `rcsb_search_url`  RCSB search URL containing encoded query conditions  
- `fasta_output_dir`  Directory to save downloaded FASTA files  

#### Outputs
- FASTA files for all PDB entries matching the search criteria → `fasta_output_dir`  
**Example:**

In [ ]:
!python data_processed/script/step1_fasta_data_crawling.py \
    --url "https://www.rcsb.org/search?request=%7B%22query%22%3A%7B%22type%22%3A%22group%22%2C%22logical_operator%22%3A%22and%22%2C%22nodes%22%3A%5B%7B%22type%22%3A%22group%22%2C%22logical_operator%22%3A%22and%22%2C%22nodes%22%3A%5B%7B%22type%22%3A%22group%22%2C%22nodes%22%3A%5B%7B%22type%22%3A%22terminal%22%2C%22service%22%3A%22text%22%2C%22parameters%22%3A%7B%22attribute%22%3A%22rcsb_entry_info.resolution_combined%22%2C%22operator%22%3A%22less_or_equal%22%2C%22negation%22%3Afalse%2C%22value%22%3A5%7D%7D%5D%2C%22logical_operator%22%3A%22and%22%7D%5D%2C%22label%22%3A%22text%22%7D%5D%7D%2C%22return_type%22%3A%22entry%22%2C%22request_options%22%3A%7B%22paginate%22%3A%7B%22start%22%3A0%2C%22rows%22%3A25%7D%2C%22results_content_type%22%3A%5B%22experimental%22%5D%2C%22sort%22%3A%5B%7B%22sort_by%22%3A%22score%22%2C%22direction%22%3A%22desc%22%7D%5D%2C%22scoring_strategy%22%3A%22combined%22%7D%2C%22request_info%22%3A%7B%22query_id%22%3A%22a4530b32148c8ce9353e8893c238f68f%22%7D%7D" \
    --fasta_save_folder data_processed/save_folder/step1_fastadownload

## Step 2: FASTA → PKL Processing
This step converts protein FASTA files into structured PKL format for downstream analysis.

### Key Features
- Supports:
  - Directory of FASTA files  
  - Single merged FASTA file  
- Parallel processing for efficiency  
- Multiple parsing modes (complex / entity / single)  
- Optional filtering by PDB ID  

### Arguments Description
 `input_fasta_dir`Path to input FASTA data (folder or single file)
 `one_pdb_output_pkl_dir`Directory for intermediate PKL files  
 `final_pkl_path`Path to save the merged PKL file  
 `mode`
Processing mode:
- `complex`: full complex  
- `entity`: split by entity  
- `single`: split into chains  
- `temporary_deleted` Whether to delete intermediate PKL files after merging  
- `pdb_id_file` Optional file with PDB IDs for filtering  

In [ ]:
!python3    data_processed/script/step2_fasta_file_processed.py \
    --input_fasta_dir data_processed/save_folder/step1_fastadownload \
    --one_pdb_output_pkl_dir data_processed/save_folder/step2/one_pdb_output_pkl_dir \
    --final_pkl_path data_processed/save_folder/step2/all_pdb_chains_fasta_data.pkl \
    --mode single \
    --temporary_deleted \
    --pdb_id_file ''

### Step 3: mmCIF → Processed PDB & Metadata
This step converts downloaded mmCIF (`.cif.gz`) files into structured PDB files and metadata for downstream modeling.

#### Functionality
- Parse mmCIF structure files  
- Extract valid protein chains  
- Filter by sequence length and chain conditions  
- Convert structures into PDB format  

#### Inputs
- `input_cif_gz_dir`  Directory containing mmCIF (`.cif.gz`) files  
- `output_pdb_dir`  Directory to save processed PDB files  
- `metadata_json_dir`  Directory to save metadata JSON files  
- `need_processed_path` (optional)  File containing PDB IDs to process (one ID per line)  

#### Processing Parameters
- `mode`  Processing mode:  
  - `single`: process individual chains  
  - `complex`: process multi-chain complexes  
- `cpu_ratio`  CPU usage ratio for parallel processing  
- `min_len`  Minimum allowed chain length  
- `max_len`  Maximum allowed chain length  
- `only_chain_num`  Filter by exact number of chains (`-1` means no restriction)  
- `distance_threshold`  Distance cutoff (Å) for defining residue contacts; use 10000 Å when merging all chains  
- `max_merge_chains`  Maximum number of chains in merged complexes  

#### Outputs
- Processed PDB files → `output_pdb_dir`  
- Metadata JSON files → `metadata_json_dir`  
- Log file for missing structures (if any)  

In [ ]:
!python data_processed/script/step3_mmcif_data_crawling.py \
    --mmcif_dir data_processed/save_folder/step3 \
    --pkl_file data_processed/save_folder/step2/all_pdb_chains_fasta_data.pkl \
    --max_retries 5 \
    --retry_threshold 0.01 \
    --cpu_usage 0.75

### Step 4: mmCIF → Processed PDB & Metadata
This step converts downloaded mmCIF (`.cif.gz`) files into structured PDB files and metadata for downstream modeling.

#### Functionality
- Parse mmCIF structure files  
- Extract valid protein chains  
- Filter by sequence length and chain conditions  
- Convert structures into PDB format  
- Generate metadata JSON files  

#### Inputs
- `input_cif_gz_dir`  Directory containing mmCIF (`.cif.gz`) files  
- `output_pdb_dir`  Directory to save processed PDB files  
- `metadata_json_dir`   Directory to save metadata JSON files  
- `need_processed_path` (optional)   File containing PDB IDs to process (one ID per line)  
#### Processing Parameters
- `mode`  Processing mode:  
  - `single`: process individual chains. If you want to customize the merging method of the chain, please use the single mode, output as a single chain, and then use the structure_mmcif_decessed_main.merge_two_pd_files method in step 4 to merge
  - `complex`: process multi-chain complexes  
- `cpu_ratio`  CPU usage ratio for parallel processing  
- `min_len`  Minimum allowed chain length  
- `max_len`   Maximum allowed chain length  
- `only_chain_num`  Filter structures by exact number of chains ,`-1` means no restriction  
- `distance_threshold`  Distance cutoff (Å) used to define residue contacts ,When dealing with composites. When merging all successfully processed chains, please use the default contact distance of 10000 angstroms ,
- `max_merge_chains`  Maximum number of chains allowed in merged complexes  
#### Outputs
- Processed PDB files → `output_pdb_dir`  
- Metadata JSON files → `metadata_json_dir`  
- Log file for missing structures (if any)  

In [ ]:
!python data_processed/script/step4_pdb_parsing/structure_mmcif_processed_main.py \
    --input_cif_gz_dir data_processed/save_folder/step3 \
    --output_pdb_dir data_processed/save_folder/step4/pdb \
    --metadata_json_dir data_processed/save_folder/step4/metadata_jsonfile_output \
    --mode 'single'


Scanning directory for mmCIF files...
Total to process: 140
Already processed: 0
Remaining: 140
Total files to process: 140
Using 87 workers.
/opt/data/private/lyb/data_processed/temp2/step3/10ju.cif.gz
100%|█████████████████████████████████████████| 140/140 [19:07<00:00,  8.19s/it]


Attention: The processed data may contain larger proteins (more than 20 chains), which can cause data processing to lag in the last few strands. If necessary, it is recommended to stop and try processing multiple times. After several treatments, the number of successfully processed protein structures remains stable to ensure that all protein structures except for long proteins can be processed,

### Step 4-1: Sequence Alignment (PDB ↔ FASTA)

This optional step aligns sequences extracted from PDB structures with sequences obtained from FASTA files.

#### Purpose
In some cases (e.g., antibody structures), PDB-derived sequences may contain **missing residues**, especially at terminal regions.

This step:
- Aligns PDB sequences with FASTA sequences  
- Removes leading/trailing missing regions  
- Produces cleaned and consistent sequences  
- Generates aligned PKL data for downstream tasks  
#### Functionality
- Extract sequences from PDB files , Match with corresponding FASTA sequences ,Perform sequence alignment , Trim inconsistent regions  
- Save:
  - Aligned sequences (PKL)  
  - Alignment metadata (JSON)  
#### Inputs
- `pdb_dir`  Directory containing processed PDB files  
- `fasta_dir`   Directory containing FASTA sequences  
#### Outputs
- Aligned PKL files → `pkl_dir`  
- Alignment metadata → `json_dir`  
- Final merged PKL file  
#### Notes
- This step is **optional but recommended** for datasets with missing residues  
- Especially useful for antibody and incomplete structures  

In [ ]:
!python data_processed/script/step4-1_pdb_seq_align_with_fasta.py \
    --pdb_dir data_processed/save_folder/step4/pdb \
    --fasta_dir data_processed/save_folder/step1_fastadownload \
    --json_dir data_processed/save_folder/step4-1_align_pdb_to_fasta/align_json_output_dir \
    --pkl_dir data_processed/save_folder/step4-1_align_pdb_to_fasta/one_pkl_output \
    --num_processes 32 \
    --temporary_deleted

/opt/conda/envs/egnn_diff/lib/python3.10/site-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(
Total alignment tasks: 648
Start alignment...
Processing Alignments: 100%|█████████████| 648/648 [00:00<00:00, 1116184.39it/s]
Merging PKL files...
all one pkl nums:596,Final merged pkl saved at: /opt/data/private/lyb/data_processed/temp2/step4-1_align_pdb_to_fasta/step4-1_final_merged.pkl
Temporary pkl files deleted.
Merge completed. Final PKL saved to: /opt/data/private/lyb/data_processed/temp2/step4-1_align_pdb_to_fasta/step4-1_final_merged.pkl


# Step 5: Representation Generation (Protenix)
In this step, we generate protein sequence representations using the Protenix framework.

## Overview
To compute protein representations, A dedicated environment with all required Protenix dependencies must be set up, and a customized version of Protenix is provided within this project. However, due to size constraints, data files and model weights are not distributed within the repository.
Please download:
- model_v0.2.0.pt from Protenix release
- CCD cache file: components.v20240608.cif,

### Step 5-1:Environment Setup
- conda create -n protenix_env python=3.10 -y
- conda activate protenix_env
- pip install protenix
- cd ../protenix

Note: This project includes a modified version of Protenix. Make sure to use this directory for execution.

### Step 5-2: MSA Generation
This step generates Multiple Sequence Alignment (MSA) data using the Protenix API.
#### Purpose
To extract protein sequences from Step 2 or Step 4-1 outputs ,or a composite fasta file containing multiple sequences and generate MSA features for downstream representation learning.
#### Functionality
- Load input sequences from:
  - FASTA file  
  - PKL file (Step 2 / Step 4-1 output)  
- Extract protein sequences, Submit sequences to Protenix MSA API, Retrieve MSA results, Save MSA files for downstream tasks  
#### Inputs
- `input_path`  Path to input file (`.fasta` or `.pkl`)  
- `out_dir`  Directory to store MSA results  
- `log_file`  Log file for tracking API requests and failures  
#### Parameters
- `max_workers`  Number of parallel API requests  
#### Outputs
- MSA files (e.g., `.a3m`, `.msa`)  
- Log file for request status  


In [ ]:
!python step5_representation_generation/step2_get_msa_from_protenixAPI.py \
    --input_path data_processed/save_folder/step4-1_align_pdb_to_fasta/step4-1_final_merged.pkl \
    --out_dir data_processed/save_folder/step5_msa_output \
    --log_file data_processed/save_folder/step5_msa_output/msa_log.log \
    --max_workers 20

`Note`: Protenix does not include a Jupyter kernel; please <b>execute the operations in the terminal</b>. When dealing with large-scale data, the speed of MSA may be relatively slow, so please exercise patience. If the MSA server frequently encounters request failures, you may consider localizing MSA and running it using ColabFold in conjunction with MMseqs2. </br>For detailed instructions, please refer to the Protenix documentation at: https://github.com/bytedance/Protenix/blob/main/docs/colabfoldcompatiblemsa.md. </br>Furthermore, after MSA generates the A3M file, please proceed with MSA Post-Processing, as outlined in Step 3 of the following documentation: https://github.com/bytedance/Protenix/blob/main/docs/msatemplatepipeline.md.

### Step 5-3: JSON Construction for Representation Input  
This step constructs structured JSON files by pairing precomputed MSA results with corresponding protein sequences.
#### Purpose  
To integrate MSA data from Step 5-2 with protein sequences from Step 2 / Step 4-1, and generate standardized JSON inputs for downstream representation learning models.
#### Functionality  
- Load input data from:  
  - FASTA file  
  - PKL file (Step 2 / Step 4-1 output)  
- Match each protein sequence with its corresponding precomputed MSA files  
- Construct per-sample JSON files containing:   Sequence information , MSA features (pairing / non-pairing)  
- Organize JSON files in a structured format  
- Merge individual JSON files into a final batch JSON  

#### Inputs  
- `input_filepath`  Path to input file (`.fasta` or `.pkl`)  
- `precomputed_msa_dirs`   Directory containing MSA results  
- `one_json_output_dir`   Directory for intermediate JSON files  

#### Parameters  
- `pairing_db`   Database used for MSA pairing (e.g., `uniref100`)  
- `need_process_PDBID_path`  Optional file specifying subset of PDB IDs to process  

#### Outputs  
- Per-sample JSON files (intermediate)  
- Merged JSON file for downstream model input  

In [ ]:
!python step5_representation_generation/step3_af3_input_json.py \
  --input_filepath data_processed/save_folder/step4-1_align_pdb_to_fasta/step4-1_final_merged_top3.pkl \
  --one_json_output_dir data_processed/save_folder/step5_msa_output/one_json \
  --precomputed_msa_dirs data_processed/save_folder/step5_msa_get/msa_dir \
  --merge_json_output_dir data_processed/save_folder/step5_msa_output/merge_json \
  --merge_json_output_name test.json \
  --pairing_db uniref100

### Step 5-4: Representation Generation

This step generates protein representations using a customized Protenix inference pipeline.

#### Purpose  
To generate high-dimensional sequence representations from the JSON inputs produced in Step 5-3, we employ a customized version of Protenix. This implementation is based on Protenix v0.3.1, utilizing the pretrained weights model-v0.2.0.pt and the CCD cache file components.v20240608.cif.
The corresponding Protenix release can be accessed via: https://github.com/bytedance/Protenix/releases/tag/v0.3.1
#### Inputs  
- JSON file generated from Step 5-3  
- Protenix inference script (`inference_demo_1.sh`)  
#### Parameters  
- `input_json`  Path to the input JSON file  
- `output_dir`  Directory to store generated representations  
- `num_gpus`  Number of GPUs used for inference  

#### Procedure  
1. Deactivate the current conda environment and activate the `protenix` environment  
2. Navigate to the Protenix working directory:  cd  `../data_processed/protenix`  
3. Edit the inference script:  
`protenix/inference_demo_1.sh`  
- Set the input JSON file path (from Step 5-3)  
- Set the output directory for representations  
- Configure the number of GPUs : nproc_per_node=X
- Keep other parameters unchanged  
4. Run the inference script:  bash inference_demo_1.sh

### Step 6: Representation Retrieval by Sequence  

This step retrieves precomputed representations based on protein sequences.

#### Purpose  
To establish a mapping between protein sequences and their corresponding representations by converting sequences into hashable keys, enabling efficient lookup and reuse of precomputed features.

#### Functionality  
- Convert each sequence (or multi-chain sequences) into a tuple-based key (`seqs_key`)  
- Build a mapping between sequence keys and PDB IDs  

#### Inputs  
- `input_pkl`  Path to input pkl file containing sequence data (`PDB_ID`, `seqs`)  

#### Outputs  
- Processed pkl file containing:  `PDB_ID`  , `seqs_key` (tuple representation of sequences)  


In [ ]:
!python step6_af3rep_corresponding_seqs.py \
  --input_pkl data_processed/temp/step4-1_align_pdb_to_fasta/step4-1_final_merged_top3.pkl \
  --output_pkl data_processed/temp/step6/alphafold3_seqres_to_index_output.pkl

Saved successfully: /opt/data/private/lyb/data_processed/temp/step6/alphafold3_seqres_to_index_output.pkl


### Final Data Preparation for Model Training  
At this stage, all data required for model training have been fully prepared, including the following components.  


#### Data Components  

1. **PKL files**  
   Containing protein sequences, chain information, and other metadata for data indexing and loading, corresponding to the model input arguments `train_csv` and `val_csv`.  

2. **PDB structure files**  
   Generated from standardized processing of mmCIF files in Step 4.

3. **AF3 representation files**  
   Including both single and pair representations, capturing sequence features and residue-level interactions, corresponding to the model input argument `complex_repr_data` (provided as file paths).  

4. **Sequence–structure mapping files**  
   Establishing the correspondence between sequences, structures, and their associated representations, corresponding to the model input argument `alphafold3_seqres_to_index`.  
